In [2]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd
class TransBTS_model_dataset(Dataset):
    def __init__(self, BrainMRIFilesPath, CSVPath, transform=None):        
        self.BrainMRIFilesPath = BrainMRIFilesPath
        self.CSVPath = CSVPath
        self.transform = transform
        self.df = pd.read_excel(self.CSVPath , engine='openpyxl')
        self.df = self.df.dropna(subset=['AGE'])
        
        self.Files = os.listdir(self.BrainMRIFilesPath)

        self.ID_To_Filename = {}
        self.IDs = []
        for file in self.Files:
            strc = file.split('-')
            ID = int(strc[0][-3:])
            if len(self.df[self.df['IXI_ID'] == ID]['AGE'].values)> 0:
                self.ID_To_Filename[int(strc[0][-3:])] = file
                self.IDs.append(int(strc[0][-3:]))

    def __len__(self):
        return len(self.IDs)

    def __getitem__(self, index):
        ID = self.IDs[index]
        filename = self.ID_To_Filename[ID]
        file_path = os.path.join(self.BrainMRIFilesPath ,filename)
        if self.transform is not None:
            image = transforms(file_path)              # returns torch.Tensor
            array = image.numpy()[0]

        # print(ID,'  ',self.df[self.df['IXI_ID'] == ID]['AGE'].values)
        
        age = self.df[self.df['IXI_ID'] == ID]['AGE'].values[0]
        return array, age
        

In [3]:
from monai.transforms import LoadImage, Compose, Spacing, Resize, EnsureChannelFirst, ScaleIntensity
from monai.transforms import (
    Compose, LoadImage, EnsureChannelFirst, Resize,
    RandFlip, RandRotate90,
    RandGaussianNoise, RandScaleIntensity, RandShiftIntensity
)
import numpy as np
# transforms = Compose([
#     LoadImage(image_only=True),
#     EnsureChannelFirst(),                   # shape: (1, H, W, D)
#     # ScaleIntensity(),                       # optional: normalize to 0-1
#     # Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"),  # resample to 1mm spacing
#     Resize(spatial_size=(128, 128, 128), mode='trilinear')  # resize to 128³
# ])



transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),

    # ➤ Soft geometric augmentations
    RandFlip(spatial_axis=[0], prob=0.2),           # Slight horizontal flip
    RandFlip(spatial_axis=[1], prob=0.2),           # Slight vertical flip
    RandRotate90(prob=0.2),                         # Gentle 90° rotation (rare)

    # ➤ Soft intensity augmentations
    RandGaussianNoise(prob=0.15, std=0.01),         # Light noise
    RandScaleIntensity(factors=0.05, prob=0.3),     # Slight contrast scale
    RandShiftIntensity(offsets=0.03, prob=0.3),     # Small brightness shift

    # ➤ Resize (preserves shape)
    Resize((128, 128, 128), mode="trilinear")
])

2025-07-24 18:20:16.788985: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-24 18:20:17.129226: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753361417.299450 2646287 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753361417.348108 2646287 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753361417.642960 2646287 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [4]:
pwd

'/home/ubuntu/Documents/TransBTS_Modified'

In [5]:
BrainMRIFilesPath = "/home/ubuntu/Documents/SynthBA/PreprocessedData/IXI_T1"
CSVPath = os.path.join('/home/ubuntu/Documents/TransBTS_Modified', 'IXI.xlsx')
ds = TransBTS_model_dataset(BrainMRIFilesPath, CSVPath , transforms)

In [32]:
allAgevalues = []
for i in range(len(ds)):
    print(i)
    ImageVolume, age = ds[i]
    allAgevalues.append(age)
    # print(age)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [26]:
allAgevalues

[26.272416153319643,
 48.05201916495551,
 27.381245722108144,
 27.014373716632445,
 37.765913757700204,
 58.65845311430527,
 46.42573579739904,
 39.397672826830934,
 67.77275838466804,
 73.56331279945243,
 28.347707049965777,
 60.91991786447639,
 67.53456536618754,
 59.742642026009584,
 73.54414784394251,
 25.713894592744694,
 55.540041067761805,
 25.741273100616016,
 43.460643394934976,
 53.67008898015058,
 33.686516084873375,
 27.38945927446954,
 25.661875427789184,
 27.835728952772072,
 81.94113620807666,
 50.573579739904176,
 60.930869267624914,
 51.75359342915811,
 28.933607118412045,
 76.78028747433265,
 68.85147159479808,
 24.621492128678987,
 74.01232032854209,
 27.822039698836413,
 60.86789869952088,
 78.5845311430527,
 59.00342231348392,
 25.582477754962355,
 41.09514031485284,
 35.07186858316222,
 37.73305954825462,
 70.33264887063655,
 54.48596851471595,
 56.93908281998631,
 54.27789185489391,
 22.570841889117045,
 37.002053388090346,
 72.59411362080766,
 54.68583162217659,

In [33]:
np.mean(allAgevalues),np.std(allAgevalues)

(48.65176410230225, 16.458276482768948)

In [5]:
df = pd.read_excel(CSVPath , engine='openpyxl')

In [ ]:
df[df['IXI_ID']   == 637]

In [ ]:
for i in range(len(ds)):
    ImageVolume, age = ds[i]
    print(ImageVolume.shape,'    ', age)
    

In [48]:
ImageVolume, age = ds[0]

In [49]:
ImageVolume.shape,age

((128, 128, 128), 26.272416153319643)

In [ ]:
with torch.no_grad():
    output = model(input_tensor)  

In [9]:
from models.TransBTS.TransBTS_downsample8x_skipconnection import TransBTS
_, model = TransBTS(_conv_repr=True, _pe_type="learned")
model = model.to(device)

In [8]:
import torch.nn as nn
import torch
loss_fn = nn.L1Loss()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)


In [10]:
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)

# Restore model and optimizer states
model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

In [11]:
loader = DataLoader(ds, batch_size=2, pin_memory= True, shuffle=True, num_workers=4)

In [12]:
best_loss = float('inf')  # for saving the best model
save_path = 'checkpoints'  # folder to store checkpoints
os.makedirs(save_path, exist_ok=True)

In [15]:
from tqdm import tqdm
epoch = 0
num_epochs = 100
model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    with tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}") as t:
        for batch in t:
            Imagevol = batch[0]
            Imagevol = Imagevol.unsqueeze(1).to(device)
            Age = batch[1].unsqueeze(1).to(device)
            # with torch.no_grad():
            output = model(Imagevol)  
            # print(output.shape,'  ',Age.shape)
            # outputs = model(Imagevol)
            loss = loss_fn(output, Age)
            # print(Age, '   ', output)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            t.set_postfix(loss=loss.item())
            
    epoch_loss = running_loss / len(loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss}")

    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss
        }, os.path.join(save_path, 'best_model.pt'))
        print(f"💾 Checkpoint saved at epoch {epoch+1} with loss {epoch_loss:.4f}")
            

Epoch 1/100: 100%|█████████████████| 282/282 [02:26<00:00,  1.92it/s, loss=10.6]


Epoch [1/100] Loss: 13.639970368748411


Epoch 2/100: 100%|█████████████████| 282/282 [02:27<00:00,  1.91it/s, loss=11.1]


Epoch [2/100] Loss: 13.621362224178338


Epoch 3/100: 100%|█████████████████| 282/282 [02:28<00:00,  1.90it/s, loss=13.7]


...

Epoch [98/100] Loss: 13.604922662603565


Epoch 99/100: 100%|████████████████| 282/282 [02:11<00:00,  2.14it/s, loss=9.78]


Epoch [99/100] Loss: 13.552123940801556


Epoch 100/100: 100%|███████████████| 282/282 [02:13<00:00,  2.11it/s, loss=6.56]

Epoch [100/100] Loss: 13.713113746050679


In [42]:
# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Trainable parameters (requires_grad = True)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")


Total parameters: 29401473
Trainable parameters: 29401473


In [14]:
model.eval()
for i in range(len(ds)):
    ImageVolume, age = ds[i]
    # print(ImageVolume.shape)
    ImageVolume = torch.from_numpy(ImageVolume).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(ImageVolume)  
    print(age, '   ', output[0][0].item())
    del ImageVolume

    del output
    torch.cuda.empty_cache()
    
    

26.272416153319643     49.51449966430664
48.05201916495551     51.69070816040039
27.381245722108144     52.057857513427734
27.014373716632445     51.693607330322266
37.765913757700204     49.75596237182617
58.65845311430527     48.935890197753906
46.42573579739904     31.457862854003906
39.397672826830934     52.185333251953125
67.77275838466804     52.275421142578125
73.56331279945243     51.96527099609375
28.347707049965777     51.790748596191406
60.91991786447639     51.77065658569336
67.53456536618754     51.61083984375
59.742642026009584     52.630210876464844
73.54414784394251     52.18634796142578
25.713894592744694     52.36048889160156
55.540041067761805     52.357666015625
25.741273100616016     45.411685943603516
43.460643394934976     52.130271911621094
53.67008898015058     49.81772994995117
33.686516084873375     52.32783508300781
27.38945927446954     50.20842742919922
25.661875427789184     47.213661193847656
27.835728952772072     43.54020690917969
81.94113620807666   

KeyboardInterrupt: 

In [ ]:
!pip install tqdm

In [20]:
model.eval()
for i in range(len(ds)):
    ImageVolume, age = ds[i]
    outputs = model(inputs)
    print(ImageVolume.shape,'    ', age)

NameError: name 'inputs' is not defined

In [ ]:
from tqdm import tqdm

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    with tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}") as t:
        for batch in t:
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)

            outputs = model(inputs)
            loss = loss_fn(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            t.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} Average Loss: {epoch_loss / len(loader):.4f}")